# UCB-FE Experiments

This notebooks provides all neccessary calculations for the UCB-FE experiments.
Logic about using ML models are implemeted in the [utils file](./utils_ucb_fe.py).

Note that you can either use pretrained models with weights we provided or train all models by your self. The whole training process will take approximately 6 hours on GPU, calculating metrics on CPU will take nearly 1 hour.

Also, you can use preprocessed dataset with all necessary fields, before calculating StratifiedAUC metric in this notebook

In [1]:
import numpy  as np
import pandas as pd
from tqdm import tqdm
import warnings

from sklearn.metrics import roc_auc_score
from utils_ucb_fe import ML, TabMModel, Lightgbm, Xgboost, Catboost, Tab_net

warnings.filterwarnings("ignore", category=DeprecationWarning) 

%load_ext autoreload
%autoreload 2

In [2]:
# Read data
train_df = pd.read_parquet('data/train_int.parquet')
test_df = pd.read_parquet('data/test_int.parquet')[:]
df_base = test_df[:]
df = test_df[:]

y_train = train_df.target
X_train = train_df.drop(['target', 'request_id', 'item_id', 'item_imps', 'item_shows'], axis=1)

y_test = test_df.target
X_test = test_df.drop(['target', 'request_id', 'item_id', 'item_imps', 'item_shows'], axis=1)

categorical_features = [ "mcat", "mcat_1", "mcat_2", "mcat_3", "mcat_4", "mcat_5", "cat_id", "item_region_id", "item_location_id" ]

In [ ]:
# Init models

tabm_ml = TabMModel(X_train, y_train, categorical_features)
lightgbm_ml = Lightgbm(X_train, y_train, categorical_features)
xgboost_ml = Xgboost(X_train, y_train, categorical_features)
catboost_ml = Catboost(X_train, y_train, categorical_features)
tabnet_ml = Tab_net(X_train, y_train, categorical_features)

model_dict = {
     'lightgbm': lightgbm_ml,
    'xgboost': xgboost_ml,
    'tabm': tabm_ml, 
    'catboost': catboost_ml, 
    'tabnet': tabnet_ml
}
model_names = model_dict.keys()

for model_name in model_names:
    ml = model_dict[model_name]
    try:
        ml._load_model_()
        print(f"Loaded pre-trained {model_name} model")
    except FileNotFoundError:
        print(f"No pre-trained model found for {model_name}. Fitting the model.")
        ml.fit()

In [ ]:
for model_name in model_names:
    ml = model_dict[model_name]
    ml.fit()

In [ ]:
def add_ucb_noise(df, ucb_columns, delta = 1.5, T_max = 1000, T_min = 1000):
    
    N = np.clip(df["item_imps"], a_min = 0.5, a_max = None)
    T = np.clip(df["item_shows"], a_min = T_min, a_max = T_max)
    
    for ucb_column in ucb_columns:
        df.loc[:, 'old_' + ucb_column] = df[ucb_column]
        df.loc[:, 'new_' + ucb_column] = np.clip(df[ucb_column] + np.sqrt(delta * np.log(T) / N), a_min = 0, a_max = 1)

add_ucb_noise(df, ucb_columns = ['ctr'])

def df_add_base_and_ucb_ctr_prediction(df, X_test, ml, model_name):
    X_test['ctr'] = df['old_ctr'].values
    ml.predict(X_test)
    df.loc[:, 'old_ctr_pred_'+model_name] = ml.ctr
    
    X_test['ctr'] = df['new_ctr'].values
    ml.predict(X_test)
    df.loc[:, 'new_ctr_pred_'+model_name] = ml.ctr

for model_name in model_names:
    ml = model_dict[model_name]
    df_add_base_and_ucb_ctr_prediction(df,  X_test, ml, model_name)


In [ ]:
df = pd.read_parquet("data/processed_data.parquet")
model_names = ['tabm', 'lightgbm', 'xgboost', 'catboost', 'tabnet']

"""
ROC-AUC & StratifiedAUC Baseline (old) & with UCB-Noise (new)
"""

class StratifiedAUC:
    """
    Weighted average of ROC-AUC metric by group, where weights are determined by the number of positive targets in each group. 
    StratifiedAUC is taken from a Yahoo article https://arxiv.org/abs/2312.05052
    """

    default_name = "StratifiedAUC"

    def __init__(
        self,
        target_column: str,
        group_column: str,
    ):
        self.target_column = target_column
        self.group_column = group_column

    def __call__(self, serp: pd.DataFrame, rank_column: str) -> float:
        if serp[self.target_column].sum() < 1:
            return np.nan

        def _fn_num(group_df: pd.DataFrame) -> float:
            if group_df[self.target_column].nunique() == 1:
                return np.nan
            roc_auc = roc_auc_score(group_df[self.target_column], group_df[rank_column])
            return roc_auc * group_df[self.target_column].sum()

        def _fn_den(group_df: pd.DataFrame) -> float:
            if group_df[self.target_column].nunique() == 1:
                return 0
            return group_df[self.target_column].sum()

        num = serp.groupby(self.group_column).apply(_fn_num).sum()
        den = serp.groupby(self.group_column).apply(_fn_den).sum()

        return num / den

    @property
    def name(self):
        return self.default_name
    

strat_auc_score = StratifiedAUC('target', 'request_id')
df_grouped = df.groupby('request_id')

data = []

for name in tqdm(model_names):
    auc_new = roc_auc_score(df['target'], df['new_ctr_pred_' + name])
    auc_old = roc_auc_score(df['target'], df['old_ctr_pred_' + name])
    strat_auc_new = df_grouped.apply(strat_auc_score, rank_column='new_ctr_pred_' + name)
    strat_auc_old = df_grouped.apply(strat_auc_score, rank_column='old_ctr_pred_' + name)
    data.append([auc_new, auc_old, strat_auc_new.mean(), strat_auc_old.mean()])

result = pd.DataFrame(
    data=data, 
    columns= ['ROC_AUC_NEW', 'ROC_AUC_OLD', 'StratifiedAUC_NEW', 'StratifiedAUC_OLD'], 
    index= model_names
)
result

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>ROC_AUC_NEW</th>
      <th>ROC_AUC_OLD</th>
      <th>StratifiedAUC_NEW</th>
      <th>StratifiedAUC_OLD</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>tabm</th>
      <td>0.636395</td>
      <td>0.646475</td>
      <td>0.600239</td>
      <td>0.620796</td>
    </tr>
    <tr>
      <th>lightgbm</th>
      <td>0.658283</td>
      <td>0.665067</td>
      <td>0.622183</td>
      <td>0.640702</td>
    </tr>
    <tr>
      <th>xgboost</th>
      <td>0.661415</td>
      <td>0.668730</td>
      <td>0.625732</td>
      <td>0.643702</td>
    </tr>
    <tr>
      <th>catboost</th>
      <td>0.654231</td>
      <td>0.662448</td>
      <td>0.620149</td>
      <td>0.638618</td>
    </tr>
    <tr>
      <th>tabnet</th>
      <td>0.628565</td>
      <td>0.643701</td>
      <td>0.601039</td>
      <td>0.627588</td>
    </tr>
  </tbody>
</table>
</div>

In [4]:
def add_position(df, model_names):
    codes, _ = pd.factorize(df['request_id'])
    df['request_id'] = codes
    df = df.sort_values(['request_id'], ascending=[True])
    
    n_requests = df['request_id'].nunique()
    pos_serp = []
    n_items = []


    for serp_x in range(n_requests):
        df_serp_x = df[df['request_id'] == serp_x]
        n_items_serp_x = df_serp_x.shape[0]
        n_items += [n_items_serp_x] * n_items_serp_x
        pos_serp += range(n_items_serp_x)

    for name in model_names:
        df = df.sort_values(['request_id', 'old_ctr_pred_' + name], ascending=[True, False])
        df['pos_old_'+ name] = pos_serp
        df = df.sort_values(['request_id', 'new_ctr_pred_' + name], ascending=[True, False])
        df['pos_new_'+ name] = pos_serp
        df['n_items'] = n_items
    
    return df

df = add_position(df, model_names)

In [5]:
def top_tail(df, name, top_n = 10, tail_m = 30):
    from_tail_to_top_df = df[(df['pos_new_'+name]< top_n) & (df['pos_old_'+name] >= tail_m) & (df['n_items'] >= tail_m)]
    from_top_to_tail_df= df[(df['pos_old_'+name]< top_n) & (df['pos_new_'+name] >= tail_m) & (df['n_items'] >= tail_m)]

    from_tail_to_top_num = from_tail_to_top_df.shape[0]
    from_top_to_tail_num  = from_top_to_tail_df.shape[0]
        
    mean_imps_new_up = from_tail_to_top_df['item_imps'].mean()
    mean_imps_old_down = from_top_to_tail_df['item_imps'].mean()

    top_num =  df[(df['pos_new_'+name]< top_n) & (df['n_items'] >= tail_m)].shape[0]
    tail_num = df[(df['pos_new_'+name] >= tail_m) & (df['n_items'] >= tail_m)].shape[0]

    return [top_n, tail_m, round(from_tail_to_top_num /tail_num * 100, 2), round(from_top_to_tail_num / top_num * 100, 2), round(mean_imps_new_up), round(mean_imps_old_down)]

data = []
index = []
for name in model_names:
    for top_n in [30, 10]:
         for tail_m in [30]:
              index.append(name)
              data.append(top_tail(df, name, top_n, tail_m))

result = pd.DataFrame(
    data=data, 
    columns= ['top_n', 'tail_m', 'from_tail_to_top_%', 'from_top_to_tail_%', 'mean_imps_from_top_to_tail', 'mean_imps_from_tail_to_top'], 
    index = index
)

result

,top_n,tail_m,from_tail_to_top_%,from_top_to_tail_%,mean_imps_from_top_to_tail,mean_imps_from_tail_to_top
tabm,30,30,34.78,5.26,1092,11579
tabm,10,30,5.30,0.10,205,1073
lightgbm,30,30,32.96,4.99,1007,14071
lightgbm,10,30,1.68,0.04,113,1871
xgboost,30,30,30.46,4.61,1581,12219
xgboost,10,30,1.46,0.12,133,811
catboost,30,30,29.11,4.41,1392,11236
catboost,10,30,1.29,0.19,132,706
tabnet,30,30,35.92,5.44,999,14488
tabnet,10,30,3.87,0.12,104,1306


In [6]:
def top_tail_with_click(df, name, m = 30, n = 10):
    """
    m tail level, n top level
    """
    
    with_click_in_tail_m = df[(df['target'] > 0) & (df['pos_old_'+name] >= m)].shape[0]
    with_click_from_tail_m_to_top_m = df[(df['target'] > 0) & (df['pos_old_'+name] >= m) & (df['pos_new_'+name] < m)].shape[0]
    with_click_from_tail_m_to_top_n = df[(df['target'] > 0)  & (df['pos_old_'+name] >= m) & (df['pos_new_'+name] < n)].shape[0]

    with_click_in_top_m = df[(df['target'] > 0) & (df['pos_old_'+name] < m)].shape[0]
    with_click_from_top_m_to_tail_m = df[(df['target'] > 0) & (df['pos_old_'+name] < m) & (df['pos_new_'+name] >= m)].shape[0]
    with_click_from_top_n_to_tail_m = df[(df['target'] > 0)  & (df['pos_old_'+name] < n) & (df['pos_new_'+name] >= m)].shape[0]
    
    return [n, m, round(100 * with_click_from_tail_m_to_top_m/with_click_in_tail_m,2), round(100 * with_click_from_tail_m_to_top_n / with_click_in_tail_m, 2),
                             round(100 * with_click_from_top_m_to_tail_m/with_click_in_top_m ,2), round(100 * with_click_from_top_n_to_tail_m / with_click_in_top_m, 2)]
   
   
data = []
index = []
for name in model_names:
    for m in [30]:
         for n in [10]:
              index.append(name)
              data.append(top_tail_with_click(df, name, m , n))

result = pd.DataFrame(
    data=data, 
    columns= ['top_n', 'tail_m', 'click_from_tail_m_to_top_m_%', 
              'click_from_tail_m_to_top_n_%', 'click_from_top_m_to_tail_m_%', 'click_from_top_n_to_tail_m_%' ], 
    index = index
)

result

,top_n,tail_m,click_from_tail_m_to_top_m_%,click_from_tail_m_to_top_n_%,click_from_top_m_to_tail_m_%,click_from_top_n_to_tail_m_%
tabm,10,30,41.20,9.86,2.94,0.02
lightgbm,10,30,40.26,2.88,2.58,0.01
xgboost,10,30,33.69,2.63,2.26,0.03
catboost,10,30,32.18,2.40,2.23,0.04
tabnet,10,30,37.38,7.15,3.07,0.02
